# Block 10 — Outliers: Partitioning + Day 1 Wrap
### Advanced Machine Learning — M&T Bank

**Dataset:** `bank_marketing_features.csv` (unchanged since Block 3). Same feature prep and train/test split as
Blocks 4-9. No new CSV comes out of this block.

**The idea:** every statistic Block 9 computed — the IQR bounds, the Z-score threshold — was computed on the
*entire* 41,188-row dataset, test set included. That's the same mistake Block 5 warned about for `StandardScaler`,
just wearing a different outfit: any number *learned from data* needs to be learned from training data only, or
the test set stops being an honest measurement of anything.

Then a fast wrap of the whole day — one throughline, eight blocks.


## Setup

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)
pd.set_option("display.max_columns", 30)

NAVY = "#251E4E"
PINK = "#FF1675"
GRAY = "#6b7280"
ORANGE = "#FF7B01"

# Paste the raw GitHub URL for bank_marketing_features.csv below, then remove the leading '#':
# df = pd.read_csv("PASTE_RAW_GITHUB_URL_HERE", sep=";")

feature_cols = [c for c in df.columns if c not in ("education", "y", "duration")]
bool_cols = [c for c in df[feature_cols].columns if df[c].dtype == bool]
y = df["y"].values

def iqr_bounds(s):
    Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
    IQR = Q3 - Q1
    return Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

# Same split reused since Block 4
Xtr, Xte, ytr, yte = train_test_split(df, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {len(Xtr)} rows. Test: {len(Xte)} rows.")


## 1. Partition Before You Peek

Block 9's IQR and Z-score bounds were both computed on `df[col]` — the full 41,188-row column. That includes every
row that later ends up in the test set. Compare those "leaky" bounds against bounds computed on the training split
only.


In [2]:
for col in ["campaign", "duration"]:
    full_lower, full_upper = iqr_bounds(df[col])
    train_lower, train_upper = iqr_bounds(Xtr[col])
    test_flag_leaky = (Xte[col] < full_lower) | (Xte[col] > full_upper)
    test_flag_correct = (Xte[col] < train_lower) | (Xte[col] > train_upper)
    disagree = (test_flag_leaky != test_flag_correct).sum()
    print(f"--- {col} ---")
    print(f"Leaky bounds (fit on all 41,188 rows):   [{full_lower:.2f}, {full_upper:.2f}]")
    print(f"Correct bounds (fit on {len(Xtr)} train rows): [{train_lower:.2f}, {train_upper:.2f}]")
    print(f"Test-set rows the two bounds disagree on: {disagree}\n")


--- campaign ---
Leaky bounds (fit on all 41,188 rows):   [-2.00, 6.00]
Correct bounds (fit on 32950 train rows): [-2.00, 6.00]
Test-set rows the two bounds disagree on: 0

--- duration ---
Leaky bounds (fit on all 41,188 rows):   [-223.50, 644.50]
Correct bounds (fit on 32950 train rows): [-221.00, 643.00]
Test-set rows the two bounds disagree on: 3



**On `campaign`, the two bounds land in exactly the same place — zero disagreement.** On `duration`, they differ
slightly (`[-223.50, 644.50]` vs. `[-221.00, 643.00]`) and 3 test-set rows get classified differently depending on
which one you used. At 41,188 rows, one 80/20 split barely moves a quantile — the leak is real, but small enough
here to be easy to miss and easy to dismiss.


In [3]:
def fit_and_score(cap_mode):
    Xtr2, Xte2, ytr2, yte2 = train_test_split(df[feature_cols].copy(), y, test_size=0.2, random_state=42, stratify=y)
    if cap_mode == "leaky":
        lower, upper = iqr_bounds(df["campaign"])
    elif cap_mode == "correct":
        lower, upper = iqr_bounds(Xtr2["campaign"])
    else:
        lower, upper = None, None
    if lower is not None:
        Xtr2["campaign"] = Xtr2["campaign"].clip(lower=max(lower, 1), upper=upper)
        Xte2["campaign"] = Xte2["campaign"].clip(lower=max(lower, 1), upper=upper)
    for c in bool_cols:
        Xtr2[c] = Xtr2[c].astype(int)
        Xte2[c] = Xte2[c].astype(int)
    numeric_to_scale = [
        "age", "campaign", "pdays", "previous",
        "emp.var.rate", "cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed",
        "education_rank", "month_sin", "month_cos", "campaign_intensity",
        "emp.var.rate_denoised", "cons.price.idx_denoised", "cons.conf.idx_denoised",
        "euribor3m_denoised", "nr.employed_denoised",
    ]
    scaler = StandardScaler()
    Xtr2[numeric_to_scale] = scaler.fit_transform(Xtr2[numeric_to_scale])
    Xte2[numeric_to_scale] = scaler.transform(Xte2[numeric_to_scale])
    model = LogisticRegression(max_iter=2000, solver="lbfgs", C=0.5)
    model.fit(Xtr2, ytr2)
    proba = model.predict_proba(Xte2)[:, 1]
    return roc_auc_score(yte2, proba)

pd.DataFrame({
    "Setup": ["No capping", "Capped — bounds fit on ALL data (leaky)", "Capped — bounds fit on TRAIN only (correct)"],
    "Test ROC-AUC": [fit_and_score("none"), fit_and_score("leaky"), fit_and_score("correct")],
}).round(4)


,Setup,Test ROC-AUC
0,No capping,0.7958
1,Capped — bounds fit on ALL data (leaky),0.7960
2,Capped — bounds fit on TRAIN only (correct),0.7960


**The AUC doesn't move here either — 0.7958 to 0.7960 across all three setups.** At this dataset's size, the
practical damage from this particular leak is close to zero. That's worth saying plainly rather than dramatizing:
this is not a story about a number that lied. It's a story about a *process* that was wrong and happened not to
cost anything yet. The same shortcut on a smaller dataset, a rarer category, a costlier deployment decision, or a
statistic more sensitive than an IQR bound (a target encoding, a learned imputation value, a fitted threshold) can
cost real accuracy — and the only way to know in advance is to have the habit already, not to check after the fact
whether this particular case got away with it.


**The rule, stated once for every case it covers:** any number learned from data — a scaler's mean and std
(Block 5), an imputed value (Block 2), an outlier bound (today) — gets learned from the training partition only,
and is then *applied* to the test partition, never re-learned from it. Partition first. Everything downstream
respects that partition, including the parts that don't look like modeling.


## 2. Day 1 Wrap: One Course-Long Question

Every block today, in one line each.


In [4]:
timeline = pd.DataFrame([
    ["Block 2", "Diagnosed missingness by mechanism (MCAR/MAR/MNAR), not just volume", "bank_marketing_clean.csv"],
    ["Block 3", "Engineered features (binning, encoding, cyclical) + PCA-denoised the macro group", "bank_marketing_features.csv"],
    ["Block 4", "Quantified duration's leakage (0.936 vs. 0.796 AUC); watched L1 vs. L2 handle redundant features", "no new CSV"],
    ["Block 5", "Found Block 4's single-split AUC (0.796) was optimistic vs. its own 5-fold mean (0.783)", "no new CSV"],
    ["Block 6", "Put a number on the accuracy trap (+1.2 pts over doing nothing); tuning barely moved AUC", "no new CSV"],
    ["Block 7", "Threshold tuning beat class-weighting on F1; four-fifths check failed regardless of threshold", "no new CSV"],
    ["Block 8", "Lift/gain: top 10% of the list catches 44% of subscribers; probabilities are honestly calibrated", "no new CSV"],
    ["Block 9", "IQR vs. Z-score disagreed 3x on skewed columns; a sentinel code (pdays=999) fooled both methods", "bank_marketing_outliers.csv"],
    ["Block 10", "The bounds themselves need to come from train only — the same discipline as Block 5's scaler", "no new CSV"],
], columns=["Block", "What it added", "Dataset artifact"])
timeline


,Block,What it added,Dataset artifact
0,Block 2,Diagnosed missingness by mechanism (MCAR/MAR/M...,bank_marketing_clean.csv
1,Block 3,"Engineered features (binning, encoding, cyclic...",bank_marketing_features.csv
2,Block 4,Quantified duration's leakage (0.936 vs. 0.796...,no new CSV
3,Block 5,Found Block 4's single-split AUC (0.796) was o...,no new CSV
4,Block 6,Put a number on the accuracy trap (+1.2 pts ov...,no new CSV
5,Block 7,Threshold tuning beat class-weighting on F1; f...,no new CSV
6,Block 8,Lift/gain: top 10% of the list catches 44% of ...,no new CSV
7,Block 9,IQR vs. Z-score disagreed 3x on skewed columns...,bank_marketing_outliers.csv
8,Block 10,The bounds themselves need to come from train ...,no new CSV


**The throughline, stated in full now that it's had all day to build:** every technique in this course was really
one repeated question, asked about a different part of the pipeline each time — *is this number telling the truth,
or just a convenient one, and did I earn the right to trust it?* Accuracy lied about performance until Block 6 gave
it a number to be compared against. One split lied about stability until Block 5 ran five of them. The default
threshold hid a recall problem until Block 7 moved it. Two outlier-detection methods disagreed by 3x until Block 9
asked why. And the very act of measuring any of that honestly depends on a partition boundary — today's last
lesson — that has to be respected before any of the other nine blocks' numbers can be trusted at all.

**Day 2 picks up with ensembles, recommendations, and time series** — same dataset discipline, same standing
question, new model families.
